# VaR Engine Pro - guided analysis
Run top to bottom. Uses the package in ../src and the cached prices in ../data (run the CLI once first, or let cell 1 download).

In [ ]:
import sys; sys.path.insert(0,'../src')
import numpy as np, pandas as pd
from var_engine import data as D, methods, backtest, analysis
prices = D.load_prices('../data/prices_2007.csv')
rets = D.log_returns(prices)
port = D.portfolio_returns(rets, D.WEIGHTS)
print(len(port), 'days', port.index.min().date(), '->', port.index.max().date())

## 1. The method gap at 99%
The Gaussian number is the one to distrust - compare it with EVT and the fitted-t.

In [ ]:
for name, fn in [('Historical', lambda r,a: methods.historical_var_es(r,a)),('Gaussian', lambda r,a: methods.gaussian_var_es(r.mean(), r.std(ddof=1), a)),('Student-t', methods.student_t_var_es)]:
    v,e = fn(port.values, 0.99); print(f'{name:12s} VaR {v*100:.2f}%  ES {e*100:.2f}%')

## 2. EVT threshold sensitivity (the known soft spot of POT)
If the EVT answer moves a lot with the threshold, say so - don't hide behind one choice.

In [ ]:
for q in (0.90, 0.925, 0.95, 0.975):
    o = methods.evt_pot_var_es(port.values, 0.99, threshold_q=q)
    print(f'threshold q={q}: xi={o["xi"]:.3f}  VaR {o["var"]*100:.2f}%  ES {o["es"]*100:.2f}%  exceedances={o["n_exceed"]}')

## 3. Which position owns the tail? (component ES, Euler)

In [ ]:
print(analysis.component_es(rets, D.WEIGHTS).to_string(index=False))

## 4. Regime honesty check
A single full-sample VaR averages away the only thing that matters: the bad regimes.

In [ ]:
print(analysis.regime_table(port).to_string(index=False))

## Defence questions (answer in your own words before this goes on a CV)
1. Why does Basel backtest VaR but set capital on ES under FRTB?
2. Your Historical 99% model lands in which Basel zone in the last 250 days - and do Kupiec and the zone agree? Why might they not?
3. What does a GPD shape xi near 0.2 mean for whether ES is even finite?
4. Why is the Acerbi-Szekely Z2 reported without a p-value here?
5. Show the yfinance column-order trap and the test that guards it.